# Sinh dữ liệu cho Uncertainty Quantification

## Vì sao có notebook này

Bốn file dưới đây được **9 notebook UQ** dùng, nhưng **không notebook nào sinh ra chúng** —
chúng do một script rời tạo ra ngày 05/08 và script đó không còn:

| File | Ai dùng |
|---|---|
| `evaluation/uq_pred_calibration.parquet` | toàn bộ `uncertainty/01–08` |
| `evaluation/uq_pred_test.parquet` | toàn bộ `uncertainty/01–08` |
| `evaluation/qr_pred_calibration.parquet` | `uncertainty/02` (CQR) |
| `evaluation/qr_pred_test.parquet` | `uncertainty/02` (CQR) |

Hệ quả: chạy lại toàn bộ repo từ dữ liệu thô thì 4 file này **không được sinh lại** — mọi kết
quả UQ treo vào file mồ côi. Notebook này vá đúng chỗ đó.

## Công thức đã dựng lại được

Đọc ngược từ `HistGB/uq_gia_co_ban.joblib` và `HistGB/uq_heso.joblib`:

| | Model A — giá cơ bản | Model B — hệ số nhân |
|---|---|---|
| Thuật toán | HistGB (cấu hình `ALGOS["HistGB"]`) | HistGB (cùng cấu hình) |
| Feature | `CAT + B_NUM` (14) | `CAT + M_NUM` (11) |
| Target | `base_price`, **log** | `target_shown_multiplier`, **thô** |
| Train trên | `split == "train"`, riêng từng tháng | như trên |
| Dự đoán trên | `calibration` **và** `test` | như trên |

Khác biệt duy nhất so với `train/01` + `train/02`: hai notebook đó **chỉ** dự đoán trên `test`,
nên không sinh được tập `calibration` mà conformal bắt buộc phải có.

> Đã kiểm chứng: nạp `uq_gia_co_ban.joblib` cũ dự đoán lại tập test cho ra đúng cột `base_pred`
> trong file cũ, lệch tối đa **1đ** trên thang ~87.000đ. Công thức trên là đúng.

## Notebook làm gì

1. Train lại 2 model UQ (6 model: 2 × 3 tháng)
2. Sinh `uq_pred_calibration.parquet` + `uq_pred_test.parquet`
3. Sinh `qr_pred_*.parquet` + `quantile_models.joblib` từ bộ 7 phân vị của `train/06`
4. **Đối chiếu với file cũ trước khi ghi đè** — nếu lệch lớn thì dừng lại xem xét

⏱️ ~8–12 phút. Chạy **sau** `train/06`.

In [1]:
import warnings, time, sys, joblib
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
import numpy as np, pandas as pd
from pathlib import Path
from _common_train import CAT, B_NUM, M_NUM, dat_categories, prep, ALGOS

pd.set_option("display.width", 200)
EVAL = Path("../evaluation"); EVAL.mkdir(exist_ok=True)
HGB = Path("../HistGB"); HGB.mkdir(exist_ok=True)
QDIR = Path("../QuantileLGBM")

# Cot phai nap: feature cua ca 2 model + target + cot mo ta di kem trong file dau ra
COT_KEM = ["target_timestamp", "requested_lag_minutes", "persistence_prediction"]
COLS = list(dict.fromkeys(
    CAT + B_NUM + M_NUM + COT_KEM +
    ["target_shown_price", "target_shown_multiplier", "latest_observed_price",
     "latest_observed_multiplier", "evaluation_month", "split"]))
COLS = [c for c in COLS if c != "latest_observed_base"]   # cot dan xuat

df = pd.read_parquet("../../data/hcm_train_ready.parquet", columns=COLS)
df = dat_categories(df)     # co dinh danh muc TREN TAP DAY DU truoc khi chia

# 2 cot dan xuat — giong het train/01
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)

THANGS = sorted(df.evaluation_month.unique())
SPLITS = ["calibration", "test"]

print(f"Nap {len(df):,} dong x {len(COLS)} cot")
print(f"Thang: {THANGS}")
print()
print(df.groupby("split", observed=True).size().to_string())
print()
print("Se sinh du doan cho:", SPLITS)

Nap 6,897,051 dong x 26 cot
Thang: ['2026-01', '2026-02', '2026-03']



split
calibration     615908
test            864360
train          4641799
validation      774984

Se sinh du doan cho: ['calibration', 'test']


## 1. Train hai model UQ

`calibration` **không** được dùng để train — đó là điều kiện để conformal có bảo đảm hữu hạn mẫu.
Model chỉ nhìn `split == "train"`.

In [2]:
CAU_HINH = [
    ("gia_co_ban", "base_price",              B_NUM, True),   # log-target
    ("heso",       "target_shown_multiplier", M_NUM, False),  # thô
]

models = {}
t_tong = time.time()
for ten, target, num, log in CAU_HINH:
    print(f"=== {ten}  (target={target}, log={log}, {len(CAT)+len(num)} feature) ===")
    models[ten] = {}
    for th in THANGS:
        tr = df[(df.evaluation_month == th) & (df.split == "train")]
        t0 = time.time()
        m = ALGOS["HistGB"]()
        y = np.log(tr[target]) if log else tr[target]
        m.fit(prep(tr, num), y)
        models[ten][th] = m
        print(f"  [{th}] n_train={len(tr):>9,} · so_cay={m.n_iter_:>3} · "
              f"val_loss={-m.validation_score_[-1]:.5f} · {time.time()-t0:5.1f}s")
    print()

joblib.dump(models["gia_co_ban"], HGB / "uq_gia_co_ban.joblib")
joblib.dump(models["heso"],       HGB / "uq_heso.joblib")
print(f"TONG {(time.time()-t_tong)/60:.1f} phut")
print(f"Da luu -> {HGB/'uq_gia_co_ban.joblib'}")
print(f"Da luu -> {HGB/'uq_heso.joblib'}")

=== gia_co_ban  (target=base_price, log=True, 14 feature) ===


  [2026-01] n_train=1,544,286 · so_cay=500 · val_loss=0.01623 ·  15.2s


  [2026-02] n_train=1,547,985 · so_cay=500 · val_loss=0.01643 ·  16.8s


  [2026-03] n_train=1,549,528 · so_cay=500 · val_loss=0.01644 ·  17.4s

=== heso  (target=target_shown_multiplier, log=False, 11 feature) ===


  [2026-01] n_train=1,544,286 · so_cay=500 · val_loss=0.00037 ·  15.2s


  [2026-02] n_train=1,547,985 · so_cay=500 · val_loss=0.00034 ·  16.6s


  [2026-03] n_train=1,549,528 · so_cay=500 · val_loss=0.00034 ·  14.2s



TONG 1.7 phut
Da luu -> ..\HistGB\uq_gia_co_ban.joblib
Da luu -> ..\HistGB\uq_heso.joblib


## 2. Sinh dự đoán trên `calibration` và `test`

**Thứ tự hàng:** gộp theo tháng, tháng tăng dần. Mọi notebook UQ đều giả định thứ tự này khi
ghép với các file khác — phải giữ nguyên.

`hybrid_pred = base_pred × heso_pred` đúng theo kiến trúc đã chốt.

In [3]:
def sinh(split):
    phan = []
    for th in THANGS:
        s = df[(df.evaluation_month == th) & (df.split == split)]
        if len(s) == 0:
            continue
        bp = np.exp(models["gia_co_ban"][th].predict(prep(s, B_NUM)))
        hp = models["heso"][th].predict(prep(s, M_NUM))
        phan.append(pd.DataFrame({
            "evaluation_month": s.evaluation_month.values,
            "split": split,
            "target_timestamp": s.target_timestamp.values,
            "gio_vn": s.gio_vn.values,
            "weather_main": s.weather_main.astype(str).values,
            "quote_distance": s.quote_distance.values,
            "quote_duration": s.quote_duration.values,
            "requested_lag_minutes": s.requested_lag_minutes.values,
            "gia_that": s.target_shown_price.values.astype("float64"),
            "heso_that": s.target_shown_multiplier.values.astype("float64"),
            "base_that": s.base_price.values.astype("float64"),
            "base_pred": bp,
            "heso_pred": hp,
            "hybrid_pred": bp * hp,
            "persistence": s.persistence_prediction.values.astype("float64"),
        }))
    return pd.concat(phan, ignore_index=True)


moi = {sp: sinh(sp) for sp in SPLITS}
for sp, d in moi.items():
    print(f"{sp:12s}: {len(d):>9,} dong x {d.shape[1]} cot")
display(moi["test"].head(3))

calibration :   615,908 dong x 15 cot
test        :   864,360 dong x 15 cot


,evaluation_month,split,target_timestamp,gio_vn,weather_main,quote_distance,quote_duration,requested_lag_minutes,gia_that,heso_that,base_that,base_pred,heso_pred,hybrid_pred,persistence
0,2026-01,test,2026-01-28 00:00:07,7,Rain,5.708,1018.409973,5,106000.0,1.18,89830.515625,87101.387739,1.152201,100358.305265,145000.0
1,2026-01,test,2026-01-28 00:00:07,7,Rain,5.708,1018.409973,10,106000.0,1.18,89830.515625,87002.542326,1.170802,101862.731497,74000.0
2,2026-01,test,2026-01-28 00:00:07,7,Rain,5.708,1018.409973,15,106000.0,1.18,89830.515625,86953.070681,1.163376,101159.145110,93000.0


## 3. Đối chiếu với file cũ — **trước khi** ghi đè

Đây là bước quan trọng nhất của notebook. Nếu bản mới lệch nhiều so với bản cũ thì mọi con số
trong báo cáo tuần 3 phải sửa; nếu lệch trong nhiễu ngẫu nhiên thì không phải sửa gì.

Điều kiện tiên quyết: **hai file phải cùng thứ tự hàng**. Kiểm bằng `gia_that` trước, nếu lệch
thì dừng — so sánh trên thứ tự khác nhau là vô nghĩa.

In [4]:
def do(y, p):
    mae = np.abs(p - y).mean()
    mape = (np.abs(p - y) / y).mean()
    r2 = 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return mae, mape, r2


rows = []
for sp in SPLITS:
    f_cu = EVAL / f"uq_pred_{sp}.parquet"
    if not f_cu.exists():
        print(f"{sp}: chua co file cu -> bo qua doi chieu")
        continue
    cu = pd.read_parquet(f_cu)
    mo = moi[sp]
    assert len(cu) == len(mo), f"So dong lech: cu {len(cu):,} vs moi {len(mo):,}"
    assert np.allclose(cu.gia_that.values, mo.gia_that.values), \
        f"[{sp}] THU TU HANG KHONG KHOP — dung so sanh!"
    y = cu.gia_that.values
    a, b = cu.hybrid_pred.values, mo.hybrid_pred.values
    ma = do(y, a); mb = do(y, b)
    rows.append({
        "Tập": sp, "n": f"{len(cu):,}",
        "MAE cũ": f"{ma[0]:,.0f}", "MAE mới": f"{mb[0]:,.0f}",
        "Chênh MAE": f"{mb[0]/ma[0]-1:+.2%}",
        "MAPE cũ": f"{ma[1]:.2%}", "MAPE mới": f"{mb[1]:.2%}",
        "R² cũ": f"{ma[2]:.4f}", "R² mới": f"{mb[2]:.4f}",
        "Tương quan": f"{np.corrcoef(a, b)[0,1]:.6f}",
    })
print("Thu tu hang khop tren ca hai tap.\n")
display(pd.DataFrame(rows))

Thu tu hang khop tren ca hai tap.



,Tập,n,MAE cũ,MAE mới,Chênh MAE,MAPE cũ,MAPE mới,R² cũ,R² mới,Tương quan
0,calibration,"615,908","17,158","17,161",+0.02%,14.74%,14.74%,0.7192,0.7192,0.999574
1,test,"864,360","18,045","18,048",+0.02%,14.74%,14.74%,0.7299,0.7297,0.999582


### Điều quan trọng hơn MAE: `q` của conformal có đổi không?

`q` là **toàn bộ** đầu ra của tầng UQ — nó quyết định độ rộng khoảng. Nếu `q` giữ nguyên thì mọi
hình và bảng UQ không cần vẽ lại.

In [5]:
MUC = [0.70, 0.80, 0.90]
rows = []
cu_c = pd.read_parquet(EVAL / "uq_pred_calibration.parquet")
cu_t = pd.read_parquet(EVAL / "uq_pred_test.parquet")


def q_cov(cal, te, m):
    e = ((cal.hybrid_pred - cal.gia_that).abs() / cal.hybrid_pred).values
    q = np.quantile(e, m)
    lo, hi = te.hybrid_pred * (1 - q), te.hybrid_pred * (1 + q)
    return q, ((te.gia_that >= lo) & (te.gia_that <= hi)).mean(), (hi - lo).mean()


for m in MUC:
    qa, ca, ra = q_cov(cu_c, cu_t, m)
    qb, cb, rb = q_cov(moi["calibration"], moi["test"], m)
    rows.append({"Mức": f"{m:.0%}",
                 "q cũ": f"{qa:.2%}", "q mới": f"{qb:.2%}",
                 "Chênh q": f"{(qb-qa)*100:+.3f} điểm",
                 "Coverage cũ": f"{ca:.2%}", "Coverage mới": f"{cb:.2%}",
                 "Rộng cũ (đ)": f"{ra:,.0f}", "Rộng mới (đ)": f"{rb:,.0f}"})
BANG_Q = pd.DataFrame(rows)
display(BANG_Q)

lech_max = max(abs(float(r["Chênh q"].split()[0])) for r in rows)
print(f"Lech q lon nhat: {lech_max:.3f} diem")
if lech_max < 0.5:
    print("=> Trong nhieu ngau nhien. Ket qua UQ trong bao cao KHONG can sua.")
else:
    print("=> LECH DANG KE — phai xem lai truoc khi ghi de.")

,Mức,q cũ,q mới,Chênh q,Coverage cũ,Coverage mới,Rộng cũ (đ),Rộng mới (đ)
0,70%,18.77%,18.78%,+0.008 điểm,69.68%,69.73%,"45,306","45,320"
1,80%,23.25%,23.25%,-0.004 điểm,79.62%,79.60%,"56,134","56,118"
2,90%,30.11%,30.09%,-0.019 điểm,89.58%,89.55%,"72,686","72,630"


Lech q lon nhat: 0.019 diem
=> Trong nhieu ngau nhien. Ket qua UQ trong bao cao KHONG can sua.


## 4. Ghi đè

Chỉ ghi sau khi mục 3 xác nhận không lệch bất thường. Từ đây `uq_pred_*.parquet` là **sản phẩm
của notebook**, không còn mồ côi.

In [6]:
for sp in SPLITS:
    f = EVAL / f"uq_pred_{sp}.parquet"
    moi[sp].to_parquet(f, index=False)
    print(f"Da ghi {f}  ({len(moi[sp]):,} dong)")

Da ghi ..\evaluation\uq_pred_calibration.parquet  (615,908 dong)


Da ghi ..\evaluation\uq_pred_test.parquet  (864,360 dong)


## 5. `qr_pred_*` — dựng lại từ bộ 7 phân vị

`train/06` đã train 21 model quantile (7 phân vị × 3 tháng) và lưu `qr_pred_da_muc_*`.
Bộ cũ `quantile_models.joblib` chỉ có 3 phân vị `q05`/`q50`/`q95` — là **tập con** của bộ mới.

Nên không cần train lại: lấy đúng 3 phân vị đó ra. Vẫn đối chiếu với file cũ trước khi ghi đè.

In [7]:
PV3 = [0.05, 0.50, 0.95]
COT3 = ["evaluation_month", "split", "gia_that", "quang_duong", "gio_vn",
        "requested_lag_minutes", "weather_main", "q05", "q50", "q95"]

rows = []
qr_moi = {}
for sp in SPLITS:
    dm = pd.read_parquet(EVAL / f"qr_pred_da_muc_{sp}.parquet")
    qr_moi[sp] = dm[COT3].copy()
    f_cu = EVAL / f"qr_pred_{sp}.parquet"
    if not f_cu.exists():
        continue
    cu = pd.read_parquet(f_cu)
    assert len(cu) == len(dm) and np.allclose(cu.gia_that.values, dm.gia_that.values), \
        f"[{sp}] hang khong khop"
    cov_cu = ((cu.gia_that >= cu.q05) & (cu.gia_that <= cu.q95)).mean()
    cov_mo = ((dm.gia_that >= dm.q05) & (dm.gia_that <= dm.q95)).mean()
    rows.append({"Tập": sp, "n": f"{len(cu):,}",
                 "Coverage q05–q95 cũ": f"{cov_cu:.2%}",
                 "mới": f"{cov_mo:.2%}",
                 "Chênh": f"{(cov_mo-cov_cu)*100:+.2f} điểm",
                 "Tương quan q05": f"{np.corrcoef(cu.q05, dm.q05)[0,1]:.4f}",
                 "Tương quan q95": f"{np.corrcoef(cu.q95, dm.q95)[0,1]:.4f}"})
display(pd.DataFrame(rows))

for sp in SPLITS:
    f = EVAL / f"qr_pred_{sp}.parquet"
    qr_moi[sp].to_parquet(f, index=False)
    print(f"Da ghi {f}  ({len(qr_moi[sp]):,} dong)")

# quantile_models.joblib = tap con 3 phan vi cua bo da muc
dm_models = joblib.load(QDIR / "quantile_models_da_muc.joblib")
con = {k: v for k, v in dm_models.items() if k[1] in PV3}
joblib.dump(con, QDIR / "quantile_models.joblib")
print(f"Da ghi {QDIR/'quantile_models.joblib'}  ({len(con)} model = 3 phan vi x 3 thang)")

,Tập,n,Coverage q05–q95 cũ,mới,Chênh,Tương quan q05,Tương quan q95
0,calibration,"615,908",89.62%,89.53%,-0.09 điểm,0.9987,0.9989
1,test,"864,360",89.18%,89.09%,-0.09 điểm,0.9985,0.9987


Da ghi ..\evaluation\qr_pred_calibration.parquet  (615,908 dong)


Da ghi ..\evaluation\qr_pred_test.parquet  (864,360 dong)


Da ghi ..\QuantileLGBM\quantile_models.joblib  (9 model = 3 phan vi x 3 thang)


## Kết luận

### Lỗ hổng đã vá

Bảy artifact trước đây không có nguồn, nay đều do notebook sinh ra:

| Artifact | Sinh bởi |
|---|---|
| `HistGB/uq_gia_co_ban.joblib` | mục 1 |
| `HistGB/uq_heso.joblib` | mục 1 |
| `evaluation/uq_pred_calibration.parquet` | mục 2 + 4 |
| `evaluation/uq_pred_test.parquet` | mục 2 + 4 |
| `evaluation/qr_pred_calibration.parquet` | mục 5 |
| `evaluation/qr_pred_test.parquet` | mục 5 |
| `QuantileLGBM/quantile_models.joblib` | mục 5 |

→ Toàn bộ repo giờ chạy được từ dữ liệu thô, không còn file mồ côi.

### Thứ tự chạy

```
model/00_chuan_bi_du_lieu
  └─ train/01 · 02 · 03 · 04 · 05
  └─ train/06_train_quantile_da_muc     (21 model quantile)
       └─ train/07_sinh_du_lieu_UQ      ← notebook này
            └─ uncertainty/01 … 08
            └─ evaluation/04 · 06 · 07
```

### Lưu ý

Model gradient boosting có tính ngẫu nhiên dù đã cố định `random_state` (thứ tự phép cộng
song song). Nên `q` mới sẽ lệch bản cũ vài phần trăm điểm — mục 3 đo chính xác mức lệch đó và
sẽ báo nếu vượt ngưỡng 0,5 điểm.